# Chapter 6 - Statistical Machine Learning


**Nama:** Aryo Zaki Abdillah  
**NIM:** 101032300060  
**Kelas:** TK47-02  
**Universitas:** Telkom University

> Reproduksi kode dan pendalaman teoretis dari *Practical Statistics for Data Scientists* (O'Reilly), edisi ke-2.
>
> Struktur topik mengikuti notebook Python chapter 1–7 pada repository resmi:
> https://github.com/gedeck/practical-statistics-for-data-scientists

## Tujuan

Chapter ini membahas metode Statistical Machine Learning yang berfokus pada teknik klasifikasi dan regresi non-parametrik.
Topik yang dibahas meliputi K-Nearest Neighbors (KNN), Decision Trees (Pohon Keputusan), Bagging dan Random Forests,
serta Boosting (Gradient Boosting / XGBoost).

## Imports dan Setup Data

Mengimpor modul Python yang diperlukan dan menyiapkan dataset breast cancer untuk pemodelan machine learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

## 1. K-Nearest Neighbors (KNN)

KNN adalah algoritma berbasis jarak yang mengklasifikasikan observasi berdasarkan kelas mayoritas dari K tetangga terdekatnya. Standardisasi variabel sangat penting karena KNN sensitif terhadap skala fitur.

In [ ]:
data_cancer = load_breast_cancer()
X = pd.DataFrame(data_cancer.data, columns=data_cancer.feature_names)
y = pd.Series(data_cancer.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Standardisasi Fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Menentukan K terbaik
k_range = range(1, 21)
acc_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    acc_scores.append(knn.score(X_test_scaled, y_test))

plt.figure(figsize=(7, 4))
plt.plot(k_range, acc_scores, marker='o', linestyle='--')
plt.xlabel("Jumlah Tetangga (K)")
plt.ylabel("Akurasi Data Uji")
plt.title("Pemilihan Nilai K pada KNN")
plt.grid(True, alpha=0.3)
plt.show()

## 2. Decision Trees (Pohon Keputusan)

Decision Trees membagi ruang fitur secara rekursif menjadi wilayah-wilayah hiper-persegi untuk memaksimalkan kemurnian kelompok. Ukuran kemurnian yang umum digunakan adalah Gini Impurity dan Entropy.

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)
print("Akurasi Decision Tree:", accuracy_score(y_test, y_pred_tree))

plt.figure(figsize=(14, 8))
plot_tree(
    tree_model,
    feature_names=data_cancer.feature_names,
    class_names=data_cancer.target_names,
    filled=True,
    rounded=True
)
plt.title("Visualisasi Decision Tree (max_depth=3)")
plt.show()

## 3. Bagging dan Random Forests

Random Forest menggabungkan teknik Bagging (Bootstrap Aggregating) dengan pemilihan fitur acak pada setiap split untuk mengurangi varians model dan mencegah overfitting pada decision tree tunggal.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print("Akurasi Random Forest:", accuracy_score(y_test, y_pred_rf))

# Visualisasi Fitur Paling Penting (Variable Importance)
importances = pd.Series(rf_model.feature_importances_, index=data_cancer.feature_names)
top_features = importances.nlargest(10)

plt.figure(figsize=(8, 4))
top_features.plot(kind='barh')
plt.xlabel("Gini Importance")
plt.title("Top 10 Feature Importance - Random Forest")
plt.gca().invert_yaxis()
plt.show()

## 4. Boosting (Gradient Boosting)

Boosting membangun ensemble pohon keputusan secara sekuensial, di mana setiap pohon baru dilatih untuk mengoreksi residual error dari pohon sebelumnya.

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)
print("Akurasi Gradient Boosting:", accuracy_score(y_test, y_pred_gb))
print("\nReport Klasifikasi Gradient Boosting:\n", classification_report(y_test, y_pred_gb))

## Interpretasi

Berdasarkan pengujian pada Chapter 6, algoritma berbasis ensemble seperti Random Forest dan Gradient Boosting secara konsisten memberikan performa akurasi yang lebih tinggi dibanding Decision Tree tunggal atau KNN. Meskipun begitu, KNN membutuhkan prapemrosesan berupa standardisasi skala fitur, sedangkan Decision Trees memberikan kemudahan visualisasi yang sangat intuitif.

## Kesimpulan Chapter 6

Statistical Machine Learning menyediakan beragam pendekatan fleksibel untuk prediksi non-linear. Pemilihan algoritma harus mempertimbangkan kompromi antara akurasi model, efisiensi komputasi, dan tingkat interpretabilitas.